In [1]:
"""
================================================================================
MATSUMOTO-IMAI (C*) KRİPTOSİSTEMİ VE LİNEERLEŞTİRME SALDIRISI
================================================================================

Bu modül, Çok Değişkenli Kuadratik (MQ) Polinom Tabanlı Kriptografi (DPT)
ailesinin "Büyük Cisim" (Big Field) yapılandırmasına ait en temel örneklerden
biri olan Matsumoto-Imai (MI / C*) sistemini ve bu sisteme karşı Jacques
Patarin tarafından 1995 yılında öne sürülen Lineerleştirme Saldırısı'nı
(Linearization Attack) uygulamalı olarak göstermektedir.

Teorik Arka Plan:
------------------
Matsumoto-Imai kriptosistemi, gizliliği bir sonlu cisim genişlemesi (field
extension) üzerinde tanımlanan tek değişkenli bir monom dönüşümüne
dayandırır. Sistemin kalbinde şu gözlem yatar:

    F_q^n uzayı ile E = F_{q^n} cisim genişlemesi, F_q-vektör uzayı olarak
    izomorftur. Bu izomorfizma sayesinde n değişkenli bir vektör, E
    cisminde tek bir eleman olarak temsil edilebilir.

Merkez dönüşüm (central map) olarak seçilen
    F(X) = X^(1 + q^theta)
şeklindeki monom dönüşümü, E cisminde q'nun kuvvetleriyle ilişkili özel bir
üs (e = 1 + q^theta) sayesinde KOLAYCA tersinir hâle gelir (gcd(e, q^n - 1) = 1
şartı sağlandığında). Bu, imza/deşifreleme tarafında hızlı bir ters görüntü
hesaplaması yapılabilmesini sağlar.

Ancak bu monom yapının E cismi üzerinde taşıdığı X^(q^i + q^j) formundaki
terimler, açık anahtar (public key) elde edilip F_q üzerine indirgendiğinde
DOĞRUDAN GÖRÜNMEZ; gizli afin dönüşümler S ve T bu cebirsel yapıyı
gizlemek amacıyla eklenir:

    P = S ∘ F ∘ T,   P: F_q^n -> F_q^n

Burada:
    - F  : Gizli merkez dönüşüm (E cismi üzerinde monom kuvvet alma),
    - S,T: Gizli, tersinir afin (doğrusal + öteleme) dönüşümler,
    - P  : Herkese açık anahtar (açık anahtar polinom sistemi).

Lineerleştirme Saldırısının Mantığı (Tez, Bölüm 3.4 - Yapısal Saldırılar):
---------------------------------------------------------------------------
Patarin'in saldırısı, F(X) = X^(1+q^theta) monomunun E cisminde şu türden
bir CEBİRSEL BAĞINTI ürettiği gözlemine dayanır: mesaj X ve şifreli metin
Y = F(X) arasında, hem X hem Y'ye göre AYRI AYRI DOĞRUSAL, fakat birlikte
BİLİNEER (X*Y çarpım terimleri içeren) denklemler yazılabilir. Bu denklemler
S ve T'nin doğrusal/afin olması sayesinde S ve T'nin bilgisi olmadan da
x_i * y_j, x_i, y_j ve sabit terimlerin bir doğrusal kombinasyonu şeklinde
ifade edilebilir hâle gelir.

Saldırının stratejisi şu şekilde özetlenebilir:
    1. Yeterli sayıda (mesaj, şifreli metin) çifti toplanarak bu bilineer
       ilişkileri temsil eden bir Macaulay tipi matris M inşa edilir
       (Tanım 3.1.15, Tez s.52).
    2. M matrisinin sağ çekirdeği (right kernel), sistemin İÇSEL olarak
       sağladığı gizli bilineer denklemleri (R(x,y) = 0) ortaya çıkarır.
    3. Saldırganın elindeki hedef şifreli metin y* bu denklemlerde yerine
       konduğunda, bilineer terimler SABİTLEŞİR ve sistem sadece x
       değişkenlerine bağlı DOĞRUSAL bir denklem sistemine indirgenir.
    4. Bu doğrusal sistem çözülerek gizli mesaj x doğrudan hesaplanabilir.

Bu saldırı, tezin 3.4 numaralı bölümünde vurgulanan temel ilkeyi somutlaştırır:
"Merkez dönüşümün cebirsel yapısı ne kadar özel ve belirginse, bu yapı açık
anahtarda tam olarak gizlenemediği için saldırganın bu yapıyı tersine
mühendislikle ortaya çıkarması o kadar kolaylaşır."

Referans: Tez, Bölüm 5 — BigField Tabanlı CDPT Kriptosistemleri
"""

from sage.all import *

# ============================================================================
# 1. HEDEF SİSTEM: MATSUMOTO-IMAI (ULTRA DETAYLI)
# ============================================================================
class MatsumotoImai:
    """
    Matsumoto-Imai (C*) çok değişkenli kuadratik kriptosisteminin uçtan uca
    (anahtar üretimi + şifreleme) referans implementasyonu.

    Bu sınıf, Bipolar İnşa (Tez, Bölüm 3.2.1) çerçevesinde tanımlanan
        P = S ∘ F ∘ T
    yapısını somutlaştırır. Burada F, cisim genişlemesi E = F_q(w) üzerinde
    tanımlı X -> X^(1+q^theta) monom dönüşümüdür; S ve T ise F_q^n üzerinde
    rastgele seçilmiş tersinir afin dönüşümlerdir.

    Matematiksel Altyapı:
    ----------------------
    - F_q: q elemanlı temel sonlu cisim (taban cisim).
    - E = F_q[X] / (irr_poly(X)): F_q üzerinde derecesi n olan indirgenemez
      bir polinomla inşa edilmiş n. dereceden cisim genişlemesi. Bu genişleme,
      F_q^n vektör uzayı ile E cismi arasında bir F_q-vektör uzay izomorfizması
      kurulmasını sağlar (bkz. Tanım 4.1.x civarı, Tez Bölüm 4).
    - e = 1 + q^theta: Merkez dönüşümün üssü. Bu üssün q^n - 1'e göre tersi
      alınabildiğinde (gcd(e, q^n-1) = 1), F(X) = X^e dönüşümü E* üzerinde
      birebir ve örtendir; tersi F^(-1)(Y) = Y^d'dir (d = e^(-1) mod q^n-1).

    Parametreler
    ----------
    q : int
        Taban sonlu cismin eleman sayısı (asal veya asal kuvveti).
    n : int
        Değişken sayısı; aynı zamanda E cisim genişlemesinin F_q üzerindeki
        derecesidir.
    theta : int
        Merkez dönüşümün üssünü belirleyen parametre (e = 1 + q^theta).
        gcd(1 + q^theta, q^n - 1) = 1 koşulunu sağlamalıdır; aksi hâlde
        merkez dönüşüm tersinir olmaz ve sistem kurulamaz.

    Öznitelikler
    ----------
    F : FiniteField
        Taban cisim GF(q).
    R : MPolynomialRing
        F üzerinde n değişkenli çok değişkenli polinom halkası (açık anahtarın
        yaşadığı halka).
    E : FiniteField
        F'nin n. dereceden cisim genişlemesi (merkez dönüşümün yaşadığı cisim).
    e, d : int
        Sırasıyla merkez dönüşümün gizli üssü ve bu üssün modüler tersi.
    S_mat, S_vec, T_mat, T_vec : Matrix, vector
        Gizli afin dönüşümlerin matris ve öteleme bileşenleri.
    Public_Key : vector
        Anahtar üretimi sonucunda elde edilen açık anahtar polinom sistemi P.
    """

    def __init__(self, q, n, theta):
        """
        Sistem parametrelerini sabitler ve gerekli cebirsel yapıları
        (taban cisim, polinom halkası, cisim genişlemesi) inşa eder.

        Ön Koşullar
        -----------
        - q asal ya da asal kuvveti olmalıdır (GF(q) tanımlı olmalı).
        - gcd(1 + q^theta, q^n - 1) = 1 sağlanmalıdır; aksi hâlde
          `inverse_mod` çağrısı hata fırlatır ve ValueError yükseltilir.
        """
        self.q = q
        self.n = n
        self.theta = theta

        # Taban sonlu cisim F_q ve bu cisim üzerinde n değişkenli
        # çok değişkenli polinom halkası R = F_q[x_0, ..., x_{n-1}].
        # Açık anahtar polinomları bu halkada yaşayacaktır.
        self.F = GF(q, 'a')
        self.R = PolynomialRing(self.F, n, 'x')
        self.vars = vector(self.R, self.R.gens())

        # --- CİSİM GENİŞLEMESİNİN İNŞASI ---
        # Merkez dönüşüm F(X) = X^e, F_q üzerinde DEĞİL, n. dereceden bir
        # cisim genişlemesi E üzerinde tanımlıdır. Bu genişleme, F_q üzerinde
        # derecesi n olan indirgenemez bir polinomla (irr_poly) inşa edilir.
        # E ile F_q^n arasındaki izomorfizma sayesinde, n bileşenli bir
        # vektör tek bir E elemanı olarak "paketlenip" merkez dönüşümden
        # geçirilebilir.
        self.R_uni = PolynomialRing(self.F, 'X')
        self.irr_poly = self.R_uni.irreducible_element(n)
        self.E = self.F.extension(self.irr_poly, 'w')
        self.R_E = PolynomialRing(self.E, 'X')

        # --- GİZLİ ÜSSÜN BELİRLENMESİ (e VE TERSİ d) ---
        # e = 1 + q^theta, Matsumoto-Imai'nin karakteristik üssüdür. Bu üssün
        # çarpımsal grup mertebesi olan (q^n - 1)'e göre tersinir olması
        # (gcd(e, q^n-1)=1), merkez dönüşümün E* üzerinde bir permütasyon
        # (birebir-örten) olmasını garanti eder. d ise bu tersin karşılığıdır
        # ve F^(-1)(Y) = Y^d şeklindeki deşifreleme/imza üretim adımında
        # kullanılır.
        self.e = 1 + q**theta
        try:
            self.d = inverse_mod(self.e, q**n - 1)
        except:
            raise ValueError("HATA: Parametreler geçersiz (GCD != 1).")

        print("\n" + "#"*80)
        print(f"### SİSTEM KURULUMU: MATSUMOTO-IMAI (C*) ###")
        print("#"*80)
        print(f"PARAMETRELER:")
        print(f" > Cisim: GF({q})")
        print(f" > Boyut (n): {n}")
        print(f" > Theta: {theta}")
        print(f" > Gizli Üs (e): 1 + q^theta = {self.e}")
        print(f" > İndirgenemez Polinom: {self.irr_poly}")
        print("-" * 80)

        # Gizli afin dönüşümlerin (S, T) matris/vektör bileşenleri ve
        # açık anahtar; anahtar üretimi tamamlanana kadar tanımsız (None)
        # bırakılır.
        self.S_mat = None; self.S_vec = None
        self.T_mat = None; self.T_vec = None
        self.Public_Key = None

    def extension_to_vector(self, element):
        """
        E cisim genişlemesindeki bir elemanı, F_q^n vektör uzayındaki
        koordinat vektörüne dönüştürür (E ~= F_q^n izomorfizmasının ters
        yönde uygulanması).

        Bu dönüşüm, merkez dönüşümün E üzerinde ürettiği sonucun (Z_sym),
        tekrar F_q üzerindeki çok değişkenli polinom sistemine (açık
        anahtara) "geri paketlenmesi" için gereklidir.

        Parametreler
        ----------
        element : E cisim elemanı
            Vektöre dönüştürülecek eleman.

        Döndürür
        -------
        vector
            F_q üzerinde uzunluğu n olan koordinat vektörü.
        """
        # Elemanın alttaki tek değişkenli polinom temsiline erişim, Sage'de
        # kullanılan cisim genişlemesi implementasyonuna göre değişebildiği
        # için birkaç farklı erişim yolu sırayla denenir.
        try: poly = element.lift()
        except:
            try: poly = element.polynomial()
            except: poly = element
        poly = self.R_uni(poly)
        coeffs = poly.list()
        # Polinomun derecesi n-1'den küçükse, eksik katsayılar 0 ile
        # tamamlanarak sabit uzunlukta (n) bir katsayı vektörü elde edilir.
        while len(coeffs) < self.n: coeffs.append(self.F(0))
        return vector(self.F, coeffs)

    def reduce_F_ideal(self, polys):
        """
        Verilen polinom vektörünü, "cisim denklemleri" (field equations)
        ideali FieldIdeal = <x_i^q - x_i> ile indirger.

        Teorik Gerekçe:
        ----------------
        F_q sonlu cisminin her elemanı x^q = x özdeşliğini sağlar (Frobenius
        sabit noktaları). Bu nedenle F_q üzerinde çalışan çok değişkenli
        polinom sistemlerinde x_i^q - x_i şeklindeki polinomlar sıfır
        polinomuna eşdeğerdir ve bu idealle indirgeme (reduce) yapılarak
        polinomların derecesi/karmaşıklığı F_q'nun doğal yapısına uygun
        şekilde sadeleştirilir. Bu adım, açık anahtar polinomlarının
        gereksiz yüksek dereceli terimlerden arındırılmış, kanonik bir
        forma getirilmesini sağlar.

        Parametreler
        ----------
        polys : iterable
            İndirgenecek polinomlar listesi/vektörü.

        Döndürür
        -------
        vector
            Cisim ideali ile indirgenmiş polinomlardan oluşan vektör.
        """
        field_eqs = [self.vars[i]**self.q - self.vars[i] for i in range(self.n)]
        FieldIdeal = self.R.ideal(field_eqs)
        return vector(self.R, [FieldIdeal.reduce(p) for p in polys])

    def generate_keys(self):
        """
        Matsumoto-Imai sisteminin gizli anahtarlarını (S, T afin dönüşümleri)
        rastgele üretir ve bu bileşenlerle Bipolar İnşa (P = S ∘ F ∘ T)
        çerçevesinde açık anahtarı (Public_Key) hesaplar.

        Algoritma Akışı:
        -----------------
        1. Rastgele tersinir S ve T afin (doğrusal + öteleme) dönüşümleri
           üretilir. Tersinirlik şartı, S ve T'nin birer bijeksiyon
           olmasını, dolayısıyla P'nin de bijektif olmasını garanti eder
           (Bipolar İnşa'nın temel gereksinimi, Tez Bölüm 3.2.1).
        2. y = T(x) hesaplanarak gizli T dönüşümü uygulanır.
        3. y vektörü, F_q^n -> E izomorfizması aracılığıyla tek bir E
           elemanı Y_sym'e "paketlenir": Y_sym = sum(y_i * w^i).
        4. Merkez dönüşüm uygulanır: Z_sym = Y_sym^e. Bu adım, sistemin
           GİZLİLİĞİNİN KAYNAĞI olan tek yönlü monom dönüşümüdür.
        5. Z_sym, cisim denklemleriyle indirgenip (Ideal_E.reduce) tekrar
           F_q üzerinde çok değişkenli bir vektöre (z_vec) geri paketlenir.
        6. Son olarak z_vec üzerine S dönüşümü uygulanarak ve cisim
           idealiyle indirgenerek nihai açık anahtar P elde edilir.

        Döndürür
        -------
        vector
            Açık anahtar polinom sistemi P = (P_0, ..., P_{n-1}).
        """
        print("\n[ANAHTAR ÜRETİMİ] Başlıyor...")

        # --- GİZLİ AFİN DÖNÜŞÜM T ---
        # T, açık anahtarı hesaplamadan önce mesaj vektörüne uygulanan
        # gizli, tersinir bir afin dönüşümdür: T(x) = T_mat * x + T_vec.
        # Tersinirlik şartı, T'nin bir bijeksiyon olmasını garanti eder;
        # aksi hâlde P dönüşümü örten/birebir olmayabilir.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible(): break
        self.T_vec = random_vector(self.F, self.n)

        # --- GİZLİ AFİN DÖNÜŞÜM S ---
        # S, merkez dönüşümün çıktısına uygulanan ikinci gizli afin
        # dönüşümdür. Bipolar inşadaki iki bağımsız gizleme katmanından
        # (S ve T) ikincisidir.
        while True:
            self.S_mat = random_matrix(self.F, self.n, self.n)
            if self.S_mat.is_invertible(): break
        self.S_vec = random_vector(self.F, self.n)

        print("\n>>> GİZLİ ANAHTAR BİLEŞENLERİ (Private Key):")
        print(f"Gizli Matris T:\n{self.T_mat}")
        print(f"Gizli Vektör t: {self.T_vec}")
        print("-" * 40)
        print(f"Gizli Matris S:\n{self.S_mat}")
        print(f"Gizli Vektör s: {self.S_vec}")
        print("=" * 80)

        # ---------------------------------------------------------------
        # AÇIK ANAHTAR HESABI: P = S ∘ F ∘ T
        # ---------------------------------------------------------------

        # ADIM (i): y = T(x) = T_mat * x + T_vec.
        # Bu, sembolik (polinom değişkenli) bir vektördür; her bileşeni
        # x_0, ..., x_{n-1} cinsinden birinci dereceden bir polinomdur.
        y_vec = self.T_mat * self.vars + self.T_vec

        # ADIM (ii): y vektörünü E cisim genişlemesi üzerinde çalışan bir
        # çok değişkenli polinom halkasına taşıyıp, F_q^n ~= E izomorfizması
        # aracılığıyla TEK BİR E elemanına paketliyoruz:
        #     Y_sym = sum_i y_i * w^i,  w = E'nin üreteci.
        # Bu paketleme, merkez dönüşümün (X -> X^e) uygulanabilmesi için
        # zorunludur; çünkü merkez dönüşüm yalnızca E cisminde tanımlıdır.
        R_E_multi = PolynomialRing(self.E, self.n, 'x')
        y_vec_E = vector(R_E_multi, [R_E_multi(p) for p in y_vec])
        w_gen = self.E.gen()
        Y_sym = sum(y_vec_E[i] * (w_gen**i) for i in range(self.n))

        # ADIM (iii): MERKEZ DÖNÜŞÜMÜN UYGULANMASI.
        # Bu satır, Matsumoto-Imai sisteminin tüm gizliliğinin dayandığı
        # tek yönlü (fakat gizli üs sayesinde tersinir) dönüşümdür:
        #     Z_sym = Y_sym^e,  e = 1 + q^theta.
        Z_sym = Y_sym ** self.e

        # ADIM (iv): Z_sym'in cisim denklemleriyle (x_i^q - x_i) indirgenmesi.
        # E üzerindeki değişkenlerin de F_q-değerli olması gerektiğinden,
        # bu indirgeme büyük dereceli terimleri sadeleştirir.
        vars_E = R_E_multi.gens()
        field_eqs_E = [vars_E[i]**self.q - vars_E[i] for i in range(self.n)]
        Ideal_E = R_E_multi.ideal(field_eqs_E)
        Z_reduced = Ideal_E.reduce(Z_sym)

        # ADIM (v): E elemanı olan Z_reduced'in her bir monom katsayısı,
        # extension_to_vector ile tekrar F_q^n koordinat vektörüne
        # "geri paketlenir" ve z_vec = (z_0, ..., z_{n-1}) elde edilir.
        # Bu, ADIM (ii)'nin ters işlemidir: E -> F_q^n.
        z_polys = [self.R(0) for _ in range(self.n)]
        for exps, coeff in Z_reduced.dict().items():
            coeff_vec = self.extension_to_vector(coeff)
            monomial = self.R.monomial(*exps)
            for k in range(self.n):
                z_polys[k] += coeff_vec[k] * monomial
        z_vec = vector(self.R, z_polys)

        # ADIM (vi): İkinci gizli afin dönüşüm S uygulanır: S(z) = S_mat*z + S_vec.
        # Ardından cisim idealiyle son bir indirgeme yapılarak açık anahtar
        # kanonik forma getirilir.
        raw_pk = self.S_mat * z_vec + self.S_vec
        self.Public_Key = self.reduce_F_ideal(raw_pk)

        print(f"\n>>> AÇIK ANAHTAR POLİNOMLARI P(x):")
        for i, p in enumerate(self.Public_Key):
            print(f" P_{i} = {p}")
        print("=" * 80)

        return self.Public_Key

    def encrypt(self, message_vec):
        """
        Açık anahtar P kullanılarak bir mesaj vektörünü şifreler.

        Şifreleme işlemi, tanım gereği açık anahtar polinomlarının mesaj
        vektöründe değerlendirilmesinden (evaluation) ibarettir:
            y = P(x) = (P_0(x), ..., P_{n-1}(x)).
        Bu işlem, gizli anahtar bilgisine (S, T, F) hiçbir şekilde ihtiyaç
        duymaz; yalnızca herkese açık P polinomları yeterlidir.

        Parametreler
        ----------
        message_vec : vector
            F_q üzerinde uzunluğu n olan düz metin (mesaj) vektörü.

        Döndürür
        -------
        vector
            F_q üzerinde uzunluğu n olan şifreli metin vektörü.
        """
        val_dict = {self.vars[i]: message_vec[i] for i in range(self.n)}
        return vector(self.F, [p.subs(val_dict) for p in self.Public_Key])


# ============================================================================
# 2. SALDIRGAN: LINEERLEŞTİRME SALDIRISI (HER ŞEYİ YAZDIRAN MOD)
# ============================================================================
class LinearizationAttack:
    """
    Matsumoto-Imai kriptosistemine karşı Patarin'in Lineerleştirme
    Saldırısı'nın (Linearization Attack) tüm ara adımları detaylı biçimde
    yazdıran  implementasyonu.

    Teorik Gerekçe:
    ----------------
    Matsumoto-Imai'nin merkez dönüşümü F(X) = X^(1+q^theta) şeklindedir.
    Bu monomun E cisim genişlemesi üzerindeki cebirsel yapısı, mesaj X ile
    şifreli metin Y = F(X) arasında öyle bir bağıntı üretir ki, bu bağıntı
    hem X'e hem Y'ye göre ayrı ayrı doğrusal, ancak birlikte ele alındığında
    bilineer (x_i * y_j türü çarpım terimleri içeren) bir denklemler kümesi
    olarak ifade edilebilir. Bu saldırı sınıfı, söz konusu gizli bilineer
    ilişkileri veriden (chosen/known plaintext çiftlerinden) istatistiksel
    olarak keşfeder ve ardından bu ilişkileri hedef bir şifreli metne
    uygulayarak orijinal mesajı geri kazanır.

    Saldırı Ortamı:
    ----------------
    Saldırının çalıştığı R_attack polinom halkası, orijinal sistemin
    R halkasından FARKLIDIR: burada hem mesaj değişkenleri (x_0,...,x_{n-1})
    hem de şifreli metin değişkenleri (y_0,...,y_{n-1}) BİRLİKTE, aynı
    halkanın değişkenleri olarak tanımlanır. Bu, bilineer x_i*y_j terimlerinin
    ifade edilebilmesi için zorunludur.

    Parametreler
    ----------
    mi_system : MatsumotoImai
        Saldırının hedef aldığı, açık anahtarı önceden üretilmiş
        Matsumoto-Imai sistemi örneği.

    Öznitelikler
    ----------
    R_attack : MPolynomialRing
        x ve y değişkenlerini birlikte barındıran saldırı halkası.
    x_vars, y_vars : tuple
        Sırasıyla mesaj ve şifreli metin değişkenlerinin bu halkadaki
        karşılıkları.
    """

    def __init__(self, mi_system):
        """
        Saldırı için gerekli olan, mesaj (x) ve şifreli metin (y)
        değişkenlerini birlikte barındıran polinom halkasını kurar.

        Ön Koşul
        --------
        mi_system parametresinin `generate_keys()` metodu daha önce
        çağrılmış olmalı ve `Public_Key` alanı dolu olmalıdır; aksi
        hâlde `encrypt()` çağrıları başarısız olur.
        """
        self.mi = mi_system
        self.F = mi_system.F
        self.n = mi_system.n

        # --- [DÜZELTME 1] DEĞİŞKEN İSİMLERİ ---
        # x ve y değişkenlerini açıkça ayırıyoruz.
        # Bu ayrım, saldırının temel varsayımını kod düzeyinde somutlaştırır:
        # x (mesaj) bilinmeyen, y (şifreli metin) ise saldırgan tarafından
        # gözlemlenebilen büyüklüktür. İkisi de aynı halkada, fakat farklı
        # roller üstlenen değişkenler olarak tanımlanır.
        x_names = [f'x{i}' for i in range(self.n)]
        y_names = [f'y{i}' for i in range(self.n)]
        all_names = x_names + y_names

        self.R_attack = PolynomialRing(self.F, names=all_names)
        self.vars_xy = self.R_attack.gens()

        self.x_vars = self.vars_xy[:self.n]
        self.y_vars = self.vars_xy[self.n:]

        print("\n" + "#"*80)
        print(f"### SALDIRI BAŞLATILIYOR: LINEARIZATION ATTACK ###")
        print("#"*80)
        print(f" [SALDIRGAN] Saldırı Halkası Değişkenleri:")
        print(f" X (Bilinmeyen Mesaj): {self.x_vars}")
        print(f" Y (Şifreli Metin):    {self.y_vars}")
        print("="*80)

    def generate_monomials(self):
        """
        Lineerleştirme saldırısında aranan genel bağıntının (R(x,y) = 0)
        temsil edileceği monom tabanını (monomial basis) oluşturur.

        Teorik Gerekçe:
        ----------------
        Patarin'in gözlemine göre, MI sisteminin cebirsel yapısı gereği
        x ve y değişkenleri arasında yalnızca şu tür terimlerden oluşan
        doğrusal bir kombinasyon sıfıra eşitlenebilir:
            - Bilineer terimler: x_i * y_j  (derece (1,1))
            - Salt x'e bağlı doğrusal terimler: x_i
            - Salt y'ye bağlı doğrusal terimler: y_j
            - Sabit terim: 1
        Kuadratik (x_i*x_j veya y_i*y_j) terimler bu bağıntıda YER ALMAZ;
        çünkü F(X) = X^(1+q^theta) monomunun diferansiyel/polar formu
        özünde bilineerdir (bkz. Tez, Önerme 3.1.17: kuadratik bir
        dönüşümün diferansiyel formu bilineerdir).

        Döndürür
        -------
        list
            Aranan bağıntıyı temsil eden monomların sıralı listesi.
            Monom sırası, sonraki adımlarda matris sütunlarıyla birebir
            eşleştirilecektir.
        """
        monoms = []
        # --- [DÜZELTME 2] MONOM SIRALAMASI ---
        # Sadece bilineer (x*y), lineer (x, y) ve sabit terimler dikkate
        # alınır; bu sıralama, ADIM 1'deki matris M'nin sütunlarını
        # tanımlayacağı için sabit ve tutarlı tutulmalıdır.

        # 1. x_i * y_j (bilineer terimler) — n^2 adet.
        for i in range(self.n):
            for j in range(self.n):
                monoms.append(self.x_vars[i] * self.y_vars[j])

        # 2. x_i (salt mesaja bağlı doğrusal terimler) — n adet.
        for i in range(self.n):
            monoms.append(self.x_vars[i])

        # 3. y_j (salt şifreli metne bağlı doğrusal terimler) — n adet.
        for j in range(self.n):
            monoms.append(self.y_vars[j])

        # 4. Sabit terim — 1 adet.
        monoms.append(self.R_attack(1))

        return monoms

    def attack(self, target_ciphertext):
        """
        Verilen hedef şifreli metne karşılık gelen gizli mesajı,
        Lineerleştirme Saldırısı adımlarını izleyerek geri kazanmaya çalışır.

        Algoritmanın Dört Ana Adımı:
        ------------------------------
        1. VERİ TOPLAMA: Rastgele (x, y=P(x)) çiftleri üretilerek, bu
           çiftlerin bilineer/doğrusal monom değerlerinden oluşan bir
           Macaulay tipi M matrisi inşa edilir.
        2. DENKLEM KEŞFİ: M'nin sağ çekirdeği (right kernel) hesaplanır.
           Çekirdekteki her taban vektörü, x ve y arasında HER ZAMAN
           (yani sistemin gizli S, T, theta parametrelerinden bağımsız
           olarak, sadece MI yapısından ötürü) geçerli olan bir
           R(x, y) = 0 bağıntısına karşılık gelir.
        3. YERİNE KOYMA: Saldırganın kırmak istediği hedef şifreli metin
           y* bu genel bağıntılarda yerine konur. Bilineer terimler bu
           sayede SABİT katsayılı doğrusal terimlere dönüşür; sonuç olarak
           yalnızca x'e bağlı, DERECE 1 (veya sabit) polinomlar elde edilir.
        4. ÇÖZÜM: Elde edilen doğrusal denklem sistemi Ax = b biçiminde
           kurulur ve çözülerek aday mesaj(lar) belirlenir; her aday,
           gerçek şifreleme fonksiyonu ile tekrar şifrelenip hedef şifreli
           metinle karşılaştırılarak doğrulanır.

        Parametreler
        ----------
        target_ciphertext : vector
            Saldırganın ele geçirdiği ve karşılık gelen düz metni bulmak
            istediği hedef şifreli metin (F_q üzerinde uzunluğu n vektör).

        Döndürür
        -------
        vector veya None
            Doğrulanmış gizli mesaj vektörü bulunursa bu vektör; aksi
            hâlde None.
        """
        print(f"\n>>> [HEDEF] KIRILACAK ŞİFRELİ METİN (y*): {target_ciphertext}")

        # --------------------------------------------------------------------
        # ADIM 1: VERİ TOPLAMA VE MACAULAY MATRİSİNİN (M) İNŞASI
        # --------------------------------------------------------------------
        # Bu adımda amaç, R(x,y) = sum(katsayı * monom) = 0 şeklindeki
        # bilinmeyen genel bağıntıyı keşfedebilmek için yeterli sayıda
        # (x, y) gözlemi toplamaktır. Her gözlem, monom değerlerinden oluşan
        # M matrisinin bir SATIRINI oluşturur; aranan katsayı vektörü ise
        # M'nin sağ çekirdeğinde yatar (M * katsayı = 0).
        print(f"\n" + "-"*40)
        print(f"ADIM 1: P/C Çiftlerinin Toplanması ve Matris M İnşası")
        print(f"-"*40)

        monoms = self.generate_monomials()
        num_unknowns = len(monoms)

        # --- [DÜZELTME 3] OVERFITTING ÖNLEMİ ---
        # Bilinmeyen sayısından (num_unknowns) önemli ölçüde fazla örnek
        # toplanması, çekirdekte yalnızca GERÇEKTEN sistemin yapısından
        # kaynaklanan (rastgele/tesadüfi olmayan) bağıntıların kalmasını
        # sağlar. Standart lineerleştirme saldırısı literatüründe önerilen
        # örnek sayısı yaklaşık (n+1)^2 civarındadır; bu, monom sayısına
        # (n^2 + 2n + 1) yakın ölçekte bir aşırı-belirlenmişlik (overdetermined
        # system) sağlayarak yanlış/rastgele çözümlerin elenmesine yardımcı olur.
        # num_samples = num_unknowns + 20  # orijinali bu ama standart
        # lineerleştirme saldırısında (n+1)(n+1) çift kullanılıyor.
        num_samples = (self.n + 1) ** 2
        print(f" > Bilinmeyen (Monom) Sayısı: {num_unknowns}")
        print(f" > Toplanacak Örnek Sayısı (Overdefined): {num_samples}")
        print(f" > Monom Listesi (İlk 10): {monoms[:10]} ...")

        M = matrix(self.F, num_samples, num_unknowns)

        print(f"\n [VERİ] Toplanan Çiftler:")
        for row in range(num_samples):
            # Rastgele bir mesaj seçilir ve gerçek açık anahtar (P) ile
            # şifrelenir. Bu (x_val, y_val) çifti, saldırganın gözlemlediği
            # "bilinen açık metin - şifreli metin" (known plaintext) çiftini
            # temsil eder.
            x_val = random_vector(self.F, self.n)
            y_val = self.mi.encrypt(x_val)

            # Her çifti yazdır.
            print(f"   Çift #{row+1}: x={x_val} -> y={y_val}")

            # Bu çiftin ürettiği monom değerleri, M matrisinin ilgili
            # satırına, generate_monomials() ile BİREBİR AYNI SIRAYLA
            # yerleştirilir (bilineer -> x-lineer -> y-lineer -> sabit).
            col = 0
            for i in range(self.n):
                for j in range(self.n): M[row, col] = x_val[i] * y_val[j]; col += 1
            for i in range(self.n): M[row, col] = x_val[i]; col += 1
            for j in range(self.n): M[row, col] = y_val[j]; col += 1
            M[row, col] = 1

        print(f"\n [MATRİS] Oluşan M Matrisi ({num_samples}x{num_unknowns}):")
        # Matris çok büyük değilse yazdır, büyükse sadece boyut bilgisi ver.
        if num_samples < 50:
            print(M.str())
        else:
            print(" (Matris çok büyük olduğu için yazdırılmadı, ama hafızada.)")

        # --------------------------------------------------------------------
        # ADIM 2: DENKLEM KEŞFİ — ÇEKİRDEK (KERNEL) HESABI
        # --------------------------------------------------------------------
        # M matrisinin sağ çekirdeği (right kernel), M * v = 0 denklemini
        # sağlayan tüm v vektörlerinin oluşturduğu vektör uzayıdır. Bu
        # uzayın her bir taban (basis) vektörü, monom listesiyle eşlenerek
        # bir R(x, y) = 0 polinom bağıntısına dönüştürülür. Bu bağıntılar,
        # ÖRNEKLERDEN BAĞIMSIZ olarak sistemin cebirsel yapısından
        # kaynaklanan, HER ZAMAN geçerli genel ilişkilerdir.
        print(f"\n" + "-"*40)
        print(f"ADIM 2: Kernel (Çekirdek) Hesabı ve Denklem Üretimi")
        print(f"-"*40)

        kernel = M.right_kernel()
        basis = kernel.basis()

        print(f" > Çekirdek Boyutu: {len(basis)}")
        print("çekirdek=", basis)

        equations = []
        for idx, vec in enumerate(basis):
            poly = self.R_attack(0)
            for i, coeff in enumerate(vec):
                if coeff != 0: poly += coeff * monoms[i]
            equations.append(poly)
            print(f"   Bulunan Genel Denklem #{idx+1} (R(x,y)): {poly} = 0")

        if not equations:
            print("!!! HATA: Hiçbir denklem bulunamadı.")
            return None

        # --------------------------------------------------------------------
        # ADIM 3: HEDEF y* DEĞERİNİN YERİNE KONULMASI (LİNEERLEŞTİRME)
        # --------------------------------------------------------------------
        # Bu adım saldırının ADINI VEREN çekirdek fikridir: genel bağıntı
        # R(x, y) = 0 içindeki y değişkenleri, saldırganın bildiği SABİT
        # y* değerleriyle değiştirildiğinde, x_i*y*_j biçimindeki bilineer
        # terimler sabit katsayılı x_i terimlerine dönüşür. Böylece x'e
        # göre KUADRATİK olmayan (yani en fazla derece-1) bir denklem
        # sistemi elde edilmiş olur — sistem "lineerleştirilmiş" olur.
        print(f"\n" + "-"*40)
        print(f"ADIM 3: Hedef y* Değerinin Yerine Konulması")
        print(f"-"*40)

        sub_dict = {self.y_vars[i]: target_ciphertext[i] for i in range(self.n)}
        print(f" > Yerine Koyma Sözlüğü: {sub_dict}")

        linear_polys = []
        for i, eq in enumerate(equations):
            p_sub = eq.subs(sub_dict)

            print(f"\n   Denklem #{i+1} Analizi:")
            print(f"     Orijinal: {eq}")
            print(f"     y* Sonrası: {p_sub}")

            # --- [DÜZELTME 4] FİLTRELEME ---
            # Yerine koyma sonrası bazı denklemler dejenere olabilir:
            #   - Eğer denklem x'i tamamen kaybedip sabit bir sayıya
            #     dönüşmüşse (derece 0), bu denklem x hakkında hiçbir bilgi
            #     taşımaz ve elenmelidir.
            #   - Eğer denklem beklenmedik şekilde x'in 1. dereceden fazla
            #     bir kuvvetini içeriyorsa (bu durum, monom tabanının
            #     eksiksiz kurulmamış olması veya sayısal bir istisna
            #     hâlinde ortaya çıkabilir), bu da doğrusal sistem kurma
            #     amacına aykırıdır ve elenmelidir.
            if p_sub.degree() == 0:
                print(f"     -> ELENDİ (Sabit sayıya dönüştü, x içermiyor)")
                continue
            if p_sub.degree() > 1:
                print(f"     -> ELENDİ (Lineer değil, x'in derecesi > 1)")
                continue

            print(f"     -> KABUL EDİLDİ (Lineer Denklem)")
            linear_polys.append(p_sub)

        print(f"\n > Toplam Kullanılacak Lineer Denklem Sayısı: {len(linear_polys)}")

        # --------------------------------------------------------------------
        # ADIM 4: LİNEER SİSTEMİN (Ax = b) KURULMASI VE ÇÖZÜMÜ
        # --------------------------------------------------------------------
        # Elde edilen doğrusal denklemler, katsayıları x_vars değişkenlerine
        # göre okunarak standart bir Ax = b matris denklemine dönüştürülür.
        # Bu adım, saldırının MQ probleminden DOĞRUSAL CEBİR problemine
        # indirgenmesini somutlaştırır — hesaplama karmaşıklığı açısından
        # kritik kazanım tam olarak burada gerçekleşir.
        print(f"\n" + "-"*40)
        print(f"ADIM 4: Lineer Sistemin (Ax = b) Çözümü")
        print(f"-"*40)

        A_rows = []
        b_vec = []
        for p in linear_polys:
            row = []
            for var in self.x_vars:
                row.append(p.monomial_coefficient(var))
            const = p.constant_coefficient()
            A_rows.append(row)
            # p(x) = 0 denklemi A*x + const = 0 biçiminde olduğundan,
            # standart Ax = b formuna taşımak için b = -const alınır.
            b_vec.append(-const)

        A = matrix(self.F, A_rows)
        b = vector(self.F, b_vec)

        print(f" [SİSTEM] Matris A:\n{A}")
        print(f" [SİSTEM] Vektör b: {b}")

        try:
            # 1. ÖZEL ÇÖZÜM (Particular Solution):
            # Ax = b sisteminin bulunabilen herhangi bir çözümü.
            x_particular = A.solve_right(b)
            print(f"\n > Özel Çözüm (x_p): {x_particular}")

            # 2. ÇEKİRDEK (Homogeneous Solution Space):
            # Sistem tam belirlenmemiş (underdetermined) olabileceğinden,
            # A'nın çekirdeği sıfırdan büyük boyutlu olabilir. Bu durumda
            # genel çözüm, özel çözüm + çekirdek elemanlarının bir
            # kombinasyonu şeklinde ifade edilir: x = x_p + sum(c_i * k_i).
            kernel_A = A.right_kernel()
            dim = kernel_A.dimension()
            print(f" > Çözüm Uzayı Boyutu (Kernel A): {dim}")
            print("monoms=", monoms)

            candidates = []
            if dim == 0:
                # Çekirdek sıfır boyutluysa çözüm tektir.
                candidates = [x_particular]
            else:
                # Çekirdek boyutu sıfırdan büyükse, F_q cisminin tüm
                # elemanlarının kombinasyonları taranarak (dim boyutlu
                # bir F_q-vektör uzayının tüm elemanları) olası TÜM
                # adaylar üretilir. Bu, F_q sonlu olduğu için sonlu ve
                # sistematik biçimde yapılabilir bir arama uzayıdır.
                from itertools import product
                coeffs_list = [self.F] * dim
                for coeffs in product(*coeffs_list):
                    offset = vector(self.F, self.n)
                    for i, c in enumerate(coeffs):
                        offset += c * kernel_A.basis()[i]
                    candidates.append(x_particular + offset)

            print(f" > Toplam Aday Sayısı: {len(candidates)}")

            # 3. DOĞRULAMA:
            # Doğrusal sistemin çözümü, ORİJİNAL doğrusal olmayan (kuadratik)
            # açık anahtar sistemi P için de geçerli olmak ZORUNDA DEĞİLDİR;
            # çünkü lineerleştirme adımı bir GEVŞETME (relaxation) içerir.
            # Bu nedenle her aday, gerçek şifreleme fonksiyonuyla tekrar
            # şifrelenip hedef şifreli metinle karşılaştırılarak KESİN
            # olarak doğrulanmalıdır.
            print(f"\n [DOĞRULAMA] Adaylar test ediliyor...")
            for idx, cand in enumerate(candidates):
                check = self.mi.encrypt(cand)
                print(f"   Aday #{idx+1}: {cand} -> Şifreli: {check} ... ", end="")
                if check == target_ciphertext:
                    print("EŞLEŞME BAŞARILI! [✓]")
                    return cand
                else:
                    print("Başarısız [X]")

            print("\n[SONUÇ] Adaylar arasından doğru şifre bulunamadı.")
            return None

        except ValueError:
            # A.solve_right(b) çağrısı, sistem tutarsız (çözümsüz) olduğunda
            # ValueError fırlatır. Bu durum, keşfedilen doğrusal denklemlerin
            # (örneğin sayısal gürültü veya yetersiz örnek sayısı nedeniyle)
            # birbiriyle çelişmesi anlamına gelebilir.
            print("\n[SONUÇ] Sistem çözümsüz (Matris tutarsız).")
            return None




In [2]:
# ============================================================================
# SENARYO
# ============================================================================
# Aşağıdaki blok, yukarıda tanımlanan sınıfları uçtan uca çalıştırarak
# saldırının pratikte nasıl işlediğini gösteren bir demo senaryosudur.
print("\n" * 2)
my_q = 4
my_n = 3
my_theta = 2

# 1. Sistemi Kur.
# Küçük parametreler (q=4, n=3), saldırının ADIMLARININ İZLENEBİLİR olması
# amacıyla bilinçli olarak seçilmiştir; gerçek uygulamalarda n çok daha
# büyük seçilir ve bu da MI sisteminin (parametreler doğru seçilmediğinde)
# neden pratikte kırılabilir olduğunu gösterir.
mi = MatsumotoImai(q=my_q, n=my_n, theta=my_theta)

# 2. Anahtarları Üret (Her şeyi yazdıracak).
pk = mi.generate_keys()

# 3. Rastgele Bir Mesaj Seç ve Şifrele.
# Bu mesaj, saldırganın BİLMEDİĞİ, saldırı sonunda geri kazanılmaya
# çalışılacak "gerçek" gizli veridir.
secret_msg = random_vector(mi.F, my_n)
print(f"\n[SENARYO] Seçilen Gizli Mesaj (Gerçek x): {secret_msg}")

captured_cipher = mi.encrypt(secret_msg)
print(f"[SENARYO] Saldırganın Elindeki Şifre (Hedef y): {captured_cipher}")

# 4. Saldırıyı Başlat.
# Saldırgan, yalnızca açık anahtara (mi.Public_Key üzerinden encrypt()
# çağrılarıyla) ve hedef şifreli metne erişebilir; gizli S, T, F bilgisine
# HİÇBİR ŞEKİLDE erişimi yoktur.
attacker = LinearizationAttack(mi)
recovered = attacker.attack(captured_cipher)

if recovered == secret_msg:
    print("\n" + "*"*80)
    print(f"TEBRİKLER! GİZLİ MESAJ BAŞARIYLA ELE GEÇİRİLDİ.")
    print(f"Gerçek: {secret_msg}")
    print(f"Bulunan: {recovered}")
    print("*"*80)
else:
    print("\nSALDIRI BAŞARISIZ OLDU.")





################################################################################
### SİSTEM KURULUMU: MATSUMOTO-IMAI (C*) ###
################################################################################
PARAMETRELER:
 > Cisim: GF(4)
 > Boyut (n): 3
 > Theta: 2
 > Gizli Üs (e): 1 + q^theta = 17
 > İndirgenemez Polinom: X^3 + X + 1
--------------------------------------------------------------------------------

[ANAHTAR ÜRETİMİ] Başlıyor...

>>> GİZLİ ANAHTAR BİLEŞENLERİ (Private Key):
Gizli Matris T:
[    a     1     1]
[    a a + 1 a + 1]
[    0     1     0]
Gizli Vektör t: (a, a, a)
----------------------------------------
Gizli Matris S:
[    0     1     a]
[a + 1     1     0]
[    0     a     a]
Gizli Vektör s: (a + 1, a, 1)
verbose 0 (3837: multi_polynomial_ideal.py, groebner_basis) Warning: falling back to very slow toy implementation.

>>> AÇIK ANAHTAR POLİNOMLARI P(x):
 P_0 = x0^2 + (a + 1)*x1^2 + (a)*x0*x2 + x1*x2 + (a + 1)*x0 + (a)*x1
 P_1 = (a + 1)*x1^2 + (a + 1)*x0*